# Trabalho: Construção e Avaliação de um Modelo Preditivo
# Integrantes: Cesar Augusto, Luiz Lazarin e Tony

In [ ]:

# 🥑 Análise e Previsão de Preços de Abacate

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
print("✅ Bibliotecas importadas com sucesso!")

# Upload no Colab (caso necessário)
if not os.path.exists('avocado.csv'):
    from google.colab import files
    print("📁 Faça upload do arquivo avocado.csv (clique no botão abaixo)")
    uploaded = files.upload()
else:
    print("✅ Arquivo avocado.csv encontrado!")


In [ ]:

# Leitura do dataset
df = pd.read_csv('avocado.csv', encoding='latin1')

# Limpeza e padronização
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['week'] = df['Date'].dt.isocalendar().week

df['region'] = df['region'].astype(str).str.strip().str.replace(' ', '_').str.lower()
df['type'] = df['type'].astype(str).str.strip().str.lower()

df['Revenue'] = df['AveragePrice'] * df['Total Volume']

print("✅ Dataset carregado e preparado!")
print("Dimensões:", df.shape)
df.head()


In [ ]:

# Informações gerais
print("Resumo estatístico:")
display(df.describe())

print("\nValores ausentes por coluna:")
print(df.isna().sum())

# Distribuição de preço médio
plt.figure(figsize=(8,5))
sns.histplot(df['AveragePrice'], bins=30, kde=True)
plt.title('Distribuição do Preço Médio do Abacate')
plt.show()

# Evolução temporal dos preços
plt.figure(figsize=(12,6))
sns.lineplot(data=df, x='Date', y='AveragePrice', hue='type')
plt.title('Evolução do Preço Médio ao Longo do Tempo')
plt.show()


In [ ]:

# Seleciona variáveis
target = 'AveragePrice'
drop_cols = ['Date', 'region', 'type']

X = df.drop(columns=[target] + [c for c in drop_cols if c in df.columns])
y = df[target]

# Codifica variáveis categóricas se existirem
for c in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))

# Divide em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Dados preparados para modelagem:")
print("Treino:", X_train.shape, " | Teste:", X_test.shape)


In [ ]:

# Treinamento com Decision Tree
model = DecisionTreeRegressor(random_state=42, max_depth=10)
model.fit(X_train, y_train)

# Predições e métricas
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print("✅ Modelo treinado com sucesso!")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")


In [ ]:

# Importância das features
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
plt.title('Top 10 Features mais Importantes')
plt.show()

print("Top 5 Features:")
print(feature_importance.head())



##  Conclusões
- O modelo foi treinado com sucesso usando uma Árvore de Decisão.  
- Foram analisadas as variáveis mais influentes no preço médio do abacate.  
- É possível fazer previsões futuras de preço com dados de novas semanas.  
